# M11.3 test — Hub 10_2 pooled hierarchical fusion inference

Plan: [`plans/milestone_11/11_huggingface_artifacts_plan.md`](../../plans/milestone_11/11_huggingface_artifacts_plan.md).

Downloads the published model [`tbhugging/gummybear_hierarchical_fusion`](https://huggingface.co/tbhugging/gummybear_hierarchical_fusion) and runs a contract-shaped forward pass on a zero tensor with shape `[1, n_lights, n_cameras, 1, H, W]` (6×36 grid at 128×128).

**Live Hub only:** always fetches from the remote repo (no Hugging Face cache fallback). Requires network; if the Hub is unreachable the download cell fails with an explicit error after a timeout.

In [1]:
from pathlib import Path
import shutil
import sys
import subprocess

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
    if ROOT == ROOT.parent:
        raise RuntimeError("Could not locate repository root containing pyproject.toml")
    ROOT = ROOT.parent

for name in ("build", "dist"):
    shutil.rmtree(ROOT / name, ignore_errors=True)
for egg in (ROOT / "src").glob("*.egg-info"):
    shutil.rmtree(egg, ignore_errors=True)

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-cache-dir",
        f"{ROOT}[dl,hf,dev]",
        "-c",
        str(ROOT / "requirements.txt"),
    ]
)

from gummybear.paths import display_path, install_display_safe_warning_paths

install_display_safe_warning_paths()

print(f"ROOT={display_path(ROOT)}")

ROOT=.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Download published weights from the Hub

In [2]:
from IPython.display import Markdown, display

from tomography_ml_huggingface import (
    DEFAULT_HUB_DOWNLOAD_TIMEOUT_S,
    download_gummybear_hierarchical_fusion,
    load_gummybear_hierarchical_fusion,
    run_hub_contract_smoke_inference,
)

HUB_ID = "tbhugging/gummybear_hierarchical_fusion"
snap = download_gummybear_hierarchical_fusion(
    hub_id=HUB_ID, timeout_s=DEFAULT_HUB_DOWNLOAD_TIMEOUT_S
)
loaded = load_gummybear_hierarchical_fusion(snap, hub_id=HUB_ID)
display(Markdown(
    f"Loaded [`{loaded.hub_id}`](https://huggingface.co/{loaded.hub_id}) from "
    f"`{display_path(loaded.snapshot_dir)}`  \n"
    f"protocol=`{loaded.config.get('protocol')}`  "
    f"architecture=`{loaded.config.get('architecture')}`  "
    f"grid={loaded.config.get('n_lights')}×{loaded.config.get('n_cameras')}  "
    f"n_params={loaded.n_params}"
))

Fetching 4 files: 100%|██████████| 4/4 [00:03<00:00,  1.20it/s]
venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Loaded [`tbhugging/gummybear_hierarchical_fusion`](https://huggingface.co/tbhugging/gummybear_hierarchical_fusion) from `<temp>/gummybear_hub_874imvjd`  
protocol=`10_2`  architecture=`pooled_gap`  grid=6×36  n_params=748038

## Contract smoke inference

Runs one forward pass on a zero tensor at the published `[I, V, C, H, W]` shape. This verifies Hub download + load + graph execution; it is **not** a held-out localisation benchmark (the full M10 joint grid is not in packaged demo data).

In [3]:
result = run_hub_contract_smoke_inference(loaded)
yp = result.y_pred
display(Markdown(
    f"**input shape** = `{result.input_shape}`  \n"
    f"**pred xyz** = `({yp[0]:.4f}, {yp[1]:.4f}, {yp[2]:.4f})`"
))

**input shape** = `(1, 6, 36, 1, 128, 128)`  
**pred xyz** = `(0.3248, -0.5488, 8.1717)`

## Notes

- This notebook tests the **published Hugging Face Hub repo only** (live download; no cache fallback). Offline mode, missing network, or a hung request fail with `HubDownloadError` after `DEFAULT_HUB_DOWNLOAD_TIMEOUT_S` (30 s).
- Published weights are **M10 Step 3 / 10_2 pooled (GAP) hierarchical** only — not the Fourier 10_2 head from the same study checkpoint.
- Meaningful localisation error requires the full M10 illumination×camera corpus at the trained 6×36 grid; the zero-tensor smoke pass checks contract shape and execution only.